In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.impute import SimpleImputer

In [2]:
df = pd.read_csv("best_data.csv")

# PCA

In [3]:
X = df.drop("Average_Bleaching", axis=1)
Y = df["Average_Bleaching"]

In [4]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [5]:
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train_raw)
X_test_imp = imputer.transform(X_test_raw)

scaler = StandardScaler()
X_train_ss = scaler.fit_transform(X_train_imp)
X_test_ss = scaler.transform(X_test_imp)

In [6]:
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train_ss)
X_test_pca = pca.transform(X_test_ss)

# OpTuna

In [7]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, StackingRegressor
from sklearn.neural_network import MLPRegressor

In [ ]:
def stacking_objective(trial):
    # Base models
    hgb = HistGradientBoostingRegressor(
        max_iter=184, max_depth=14, learning_rate=0.0428,
        min_samples_leaf=7, l2_regularization=0.5279, random_state=42
    )
    et = ExtraTreesRegressor(
        n_estimators=380, max_depth=24, min_samples_leaf=1,
        max_features='log2', n_jobs=-1, random_state=42
    )
    xgb_m = xgb.XGBRegressor(
        n_estimators=289, learning_rate=0.0298, max_depth=9,
        subsample=0.6165, colsample_bytree=0.9922,
        n_jobs=-1, random_state=42
    )

    # Meta model (MLP) — Optuna tune นี่อย่างเดียว
    n_layers = trial.suggest_int("n_layers", 1, 3)
    layers = tuple([
        trial.suggest_int(f"n_units_l{i}", 32, 256) for i in range(n_layers)
    ])
    mlp = MLPRegressor(
        hidden_layer_sizes=layers,
        activation=trial.suggest_categorical("activation", ["relu", "tanh"]),
        learning_rate_init=trial.suggest_float("learning_rate_init", 0.0001, 0.01),
        max_iter=2000,         
        early_stopping=True,  
        n_iter_no_change=20,
        random_state=42
    )

    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=0.95)),
        ('stacking', StackingRegressor(
            estimators=[
                ('hgb', hgb),
                ('et',  et),
                ('xgb', xgb_m)
            ],
            final_estimator=mlp,
            cv=4,
            n_jobs=-1
        ))
    ])

    scores = cross_val_score(
        pipeline,
        X_train_raw,
        y_train,
        cv=4,
        scoring='r2',
    )
    return scores.mean()

stacking_study = optuna.create_study(direction="maximize")
stacking_study.optimize(stacking_objective, n_trials=50)
print("========== STACKING BEST ==========")
print(stacking_study.best_params)
print(f"Best R²: {stacking_study.best_value:.4f}")

[I 2026-05-04 20:38:35,606] A new study created in memory with name: no-name-00497a1b-67f8-4b55-a101-aca7480a1c5f
[I 2026-05-04 20:39:12,912] Trial 0 finished with value: 0.27275944731857504 and parameters: {'n_layers': 2, 'n_units_l0': 36, 'n_units_l1': 69, 'activation': 'tanh', 'learning_rate_init': 0.005118484345628773}. Best is trial 0 with value: 0.27275944731857504.
[I 2026-05-04 20:40:12,524] Trial 1 finished with value: 0.3102585212744199 and parameters: {'n_layers': 2, 'n_units_l0': 63, 'n_units_l1': 195, 'activation': 'relu', 'learning_rate_init': 0.0016213450190490685}. Best is trial 1 with value: 0.3102585212744199.
[I 2026-05-04 20:40:48,904] Trial 2 finished with value: 0.278486972949845 and parameters: {'n_layers': 3, 'n_units_l0': 145, 'n_units_l1': 53, 'n_units_l2': 138, 'activation': 'tanh', 'learning_rate_init': 0.0028978362892500545}. Best is trial 1 with value: 0.3102585212744199.
[I 2026-05-04 20:41:21,510] Trial 3 finished with value: 0.28096778096708486 and para

========== STACKING BEST ==========
{'n_layers': 1, 'n_units_l0': 49, 'activation': 'relu', 'learning_rate_init': 0.00036372857422406447}
Best R²: 0.3269


# Start

In [8]:
hgb_model = HistGradientBoostingRegressor(
    max_iter=184,
    max_depth=14,
    learning_rate=0.0428,
    min_samples_leaf=7,
    l2_regularization=0.5279,
    random_state=42
)

et_model = ExtraTreesRegressor(
    n_estimators=380,
    max_depth=24,
    min_samples_leaf=1,
    max_features='log2',
    n_jobs=-1,
    random_state=42
)

xgb_model = xgb.XGBRegressor(
    n_estimators=289,
    learning_rate=0.0298,
    max_depth=9,
    subsample=0.6165,
    colsample_bytree=0.9922,
    n_jobs=-1,
    random_state=42
)

mlp_model = MLPRegressor(
    hidden_layer_sizes=(49,),
    activation='relu',
    learning_rate_init=0.000364,
    max_iter=2000,          
    early_stopping=True,    
    n_iter_no_change=20,    
    random_state=42
)

In [9]:
stacking = StackingRegressor(
    estimators=[
        ('hgb', hgb_model),
        ('et',  et_model),
        ('xgb', xgb_model)
    ],
    final_estimator=mlp_model,
    cv=4
)

stacking.fit(X_train_pca, y_train)

,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.","[('hgb', ...), ('et', ...), ...]"
,"final_estimator final_estimator: estimator, default=NoneA regressor which will be used to combine the base estimators.The default regressor is a :class:`~sklearn.linear_model.RidgeCV`.",MLPRegressor(...ndom_state=42)
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",4
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary ` for more details.",None
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
,"verbose verbose: int, default=0Verbosity level.",0
,"loss loss: {'squared_error', 'absolute_error', 'gamma', 'poisson', 'quantile'}, default='squared_error'The loss function to use in the boosting process. Note that the""squared error"", ""gamma"" and ""poisson"" losses actually implement""half least squares loss"", ""half gamma deviance"" and ""half poissondeviance"" to simplify the computation of the gradient. Furthermore,""gamma"" and ""poisson"" losses internally use a log-link, ""gamma""requires ``y > 0`` and ""poisson"" requires ``y >= 0``.""quantile"" uses the pinball loss... versionchanged:: 0.23 Added option 'poisson'... versionchanged:: 1.1 Added option 'quantile'... versionchanged:: 1.3 Added option 'gamma'.",'squared_error'
,"quantile quantile: float, default=NoneIf loss is ""quantile"", this parameter specifies which quantile to be estimatedand must be between 0 and 1.",None
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.0428
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees.",184
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31


In [10]:
preds = stacking.predict(X_test_pca)
r2   = r2_score(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print(f"\n--- Single Split Results ---")
print(f"R-squared : {r2:.4f}")
print(f"RMSE      : {rmse:.4f}")


--- Single Split Results ---
R-squared : 0.4728
RMSE      : 7.0332


# K-Fold

In [11]:
from sklearn.model_selection import KFold, cross_val_score

In [12]:
n_splits = 4
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

r2_scores   = []
rmse_scores = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
    X_train_raw, X_test_raw = X.iloc[train_idx], X.iloc[test_idx]
    y_train,     y_test     = Y.iloc[train_idx], Y.iloc[test_idx]

    # Impute
    imputer     = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(X_train_raw)
    X_test_imp  = imputer.transform(X_test_raw)

    # Scale
    scaler      = StandardScaler()
    X_train_ss  = scaler.fit_transform(X_train_imp)
    X_test_ss   = scaler.transform(X_test_imp)

    # PCA
    pca         = PCA(n_components=0.95)
    X_train_pca = pca.fit_transform(X_train_ss)
    X_test_pca  = pca.transform(X_test_ss)

    # Stacking — สร้างใหม่ทุก fold
    stacking = StackingRegressor(
        estimators=[
            ('hgb', hgb_model),
            ('et',  et_model),
            ('xgb', xgb_model)
        ],
        final_estimator=mlp_model,
        cv=4
    )
    stacking.fit(X_train_pca, y_train)

    # Evaluate
    preds = stacking.predict(X_test_pca)
    r2_scores.append(r2_score(y_test, preds))
    rmse_scores.append(np.sqrt(mean_squared_error(y_test, preds)))

    print(f"Fold {fold}  →  R²: {r2_scores[-1]:.4f}  |  RMSE: {rmse_scores[-1]:.4f}")

print(f"\n--- {n_splits}-Fold Summary ---")
print(f"R²   mean: {np.mean(r2_scores):.4f}  ±  {np.std(r2_scores):.4f}")
print(f"RMSE mean: {np.mean(rmse_scores):.4f}  ±  {np.std(rmse_scores):.4f}")

Fold 1  →  R²: 0.3367  |  RMSE: 7.6333
Fold 2  →  R²: 0.2725  |  RMSE: 9.2757
Fold 3  →  R²: 0.2106  |  RMSE: 8.6520
Fold 4  →  R²: 0.3155  |  RMSE: 6.9131

--- 4-Fold Summary ---
R²   mean: 0.2838  ±  0.0482
RMSE mean: 8.1185  ±  0.9099
